# Configuration Loading

In every notebook so far we have constructed `Config` by hand — `Config(cwd=..., approval=...)` — passing keyword arguments at construction time. That is fine for demos and tests, but a production agent needs settings that survive across runs: the model to use, the approval policy, the working directory, developer instructions, hook scripts, MCP servers. Hardcoding these in a constructor call means editing the source every time a setting changes.

This notebook adds the configuration-loading layer from the upstream `ai-coding-agent` project. The loader reads a [TOML](https://toml.io) file from disk, layers project-level overrides on top of system-level defaults, and folds in an `AGENT.MD` file of developer instructions when one is present. The output is still the same validated [Pydantic `Config`](/notebooks/apps/cda/01-client.html) — the loader is just a constructor with a file-backed default argument.

## Where Configs Live


Two scopes are supported, mirroring the convention used by tools like `git`:

1. **System config** — one per-user file at `<user_config_dir>/ai-agent/config.toml`. On macOS this is `~/Library/Application Support/ai-agent/config.toml`; on Linux it is `~/.config/ai-agent/config.toml`; on Windows it is `%APPDATA%\ai-agent\config.toml`.
2. **Project config** — a `config.toml` inside a `.ai-agent/` directory at the agent's working directory. Project values override system values; the two are *deep-merged*.

User-config and user-data paths come from the `platformdirs` library, which knows the OS conventions so the loader does not have to:

In [ ]:
from notebooks.agent.config_loader import (
    get_config_dir,
    get_data_dir,
    get_system_config_path,
    get_project_config_path,
    get_agent_md_path,
)
import notebooks.agent.config_loader as cl

print(f"config dir : {get_config_dir()}")
print(f"data dir   : {get_data_dir()}")
print(f"system path: {get_system_config_path()}")
print(f"platform   : {cl.APP_NAME!r}  (config file name: {cl.CONFIG_FILE_NAME!r})")

**`get_config_dir` and `get_data_dir`** are thin wrappers over `platformdirs.user_config_dir` and `platformdirs.user_data_dir`. The config directory holds `config.toml`; the data directory is reserved for runtime artifacts — sessions, checkpoints, logs — kept separate from human-edited settings. Single-tenant per user, no shared `/etc` style path.

**Project config is opt-in.** `get_project_config_path` returns `None` unless a `.ai-agent/` directory exists at the cwd:

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as d:
    cwd = Path(d)
    print(f"no .ai-agent     : {get_project_config_path(cwd)}")
    (cwd / ".ai-agent").mkdir()
    print(f"empty .ai-agent  : {get_project_config_path(cwd)}")
    (cwd / ".ai-agent" / "config.toml").write_text("# placeholder\n")
    print(f"with config.toml : {get_project_config_path(cwd)}")

# And AGENT.MD is discovered the same way.
with tempfile.TemporaryDirectory() as d:
    cwd = Path(d)
    print(f"\nno AGENT.MD      : {get_agent_md_path(cwd)}")
    (cwd / "AGENT.MD").write_text("# project guide\n")
    print(f"with AGENT.MD     : {get_agent_md_path(cwd)}")

## The TOML Vocabulary


A config file is plain TOML whose keys mirror the Pydantic fields of `Config`. Nested models are written as nested tables. This is a representative file covering every top-level field the loader knows about:

```toml
[model]
name = "anthropic/claude-sonnet-4"
temperature = 1.0
context_window = 200_000

cwd = "/Users/me/projects/my-app"
approval = "on-request"
max_turns = 50

[shell_environment]
ignore_default_excludes = false
exclude_patterns = ["*KEY*", "*TOKEN*", "*SECRET*"]

developer_instructions = "This project uses Ruff for linting."
user_instructions = "Keep answers short."

[[hooks]]
name = "pre-tool-log"
trigger = "before_tool"
command = "echo $AI_AGENT_TOOL_NAME >> .ai-agent/tool.log"
hooks_enabled = true
```

Anything omitted falls back to the Pydantic defaults — partial configs are legal. `[[hooks]]` is TOML's *array of tables* syntax: each `[[hooks]]` block appends one `HookConfig` to the list (covered in [NB06](/notebooks/apps/cda/06-hooks.html)).

## Parsing TOML


Python 3.11+ ships `tomllib` in the standard library, so the loader parses TOML with no third-party dependency. `_parse_toml` wraps the two failure modes — corrupt TOML and unreadable files — into a single `ConfigError`:

In [ ]:
import tempfile
from pathlib import Path
from notebooks.agent.config_loader import _parse_toml, ConfigError

with tempfile.NamedTemporaryFile("w", suffix=".toml", delete=False) as f:
    f.write('title = "demo"\n[model]\nname = "claude-sonnet-4"\ntemperature = 0.5\n')
    ok_path = Path(f.name)
print("valid TOML ->", _parse_toml(ok_path))

with tempfile.NamedTemporaryFile("w", suffix=".toml", delete=False) as f:
    f.write('this is = = not toml')
    bad_path = Path(f.name)
try:
    _parse_toml(bad_path)
except ConfigError as e:
    print(f"bad TOML -> ConfigError: {type(e).__name__}")

**`_parse_toml` never raises `tomllib.TOMLDecodeError` or `OSError` to its caller.** It converts both into `ConfigError` so the higher layers only need to handle one exception type. The same pattern will let `load_config` decide to *skip* a corrupt file rather than abort a session — a broken project config should not block a working session.

## Deep Merging


Project config overlays system config. A flat replacement (`{**system, **project}`) would drop every key the project file omits — e.g. a project file that sets only `[model].temperature` would erase the system-level `model.name`. The loader therefore deep-merges: dicts are merged recursively, scalars replace.

In [ ]:
from notebooks.agent.config_loader import _merge_dicts

system = {
    "model": {"name": "claude-sonnet-4", "temperature": 1.0},
    "max_turns": 100,
    "approval": "on-request",
}
project = {
    "model": {"temperature": 0.4},            # override nested key
    "max_turns": 50,                          # override scalar
}
merged = _merge_dicts(system, project)

print("merged model:", merged["model"])
print("max_turns   :", merged["max_turns"])
print("approval    :", merged["approval"], "  (kept from system)")

**Deep merge semantics.** `model.name` survives because `model` is a dict in both inputs — the recursive case applies. `max_turns` is a scalar in the override, so it wins outright. `approval` only appears in `system`, so it is carried through unchanged. The implementation is a handful of lines:

In [ ]:
import inspect
from notebooks.agent import config_loader

print(inspect.getsource(config_loader._merge_dicts))

## `AGENT.MD` as Developer Instructions


Beyond typed settings, every agent benefits from free-form project context: style preferences, framework versions, lint commands, gotchas. The upstream convention is a plain `AGENT.MD` file at the working directory (this very repo uses `AGENTS.md` with the same idea). If the merged config does not already supply `developer_instructions` and an `AGENT.MD` file is present, the loader reads its full text into `Config.developer_instructions`:

In [ ]:
with tempfile.TemporaryDirectory() as d:
    cwd = Path(d)
    (cwd / "AGENT.MD").write_text(
        "# Project Guide\n\n"
        "- Python 3.13, `uv` for deps\n"
        "- Run `ruff check .` before committing\n"
        "- Never edit generated `.html` files\n"
    )
    from notebooks.agent.config_loader import load_config

    cfg = load_config(cwd=cwd)
    print(f"developer_instructions set: {cfg.developer_instructions is not None}")
    print(f"chars                       : {len(cfg.developer_instructions or '')}")
    print()
    print(cfg.developer_instructions.rstrip())

:::{.callout-note}
The `AGENT.MD` content becomes part of the system prompt that the agent sends to the LLM every turn (see the `developer_instructions` field of `Config` and the system prompt assembly in [NB03](/notebooks/apps/cda/03-agent.html)). Keep it short and high-signal — every token of instructions is a token the agent pays for on each request.

:::

:::{.callout-warning}
An explicit `developer_instructions` key in either TOML file *overrides* the `AGENT.MD` content. The loader checks the merged dict, not the `Config` field, so a project that wants to ignore an `AGENT.MD` file can set `developer_instructions = ""` in its `config.toml`.

:::

## End-to-End `load_config`


Pulling the pieces together — system + project + `AGENT.MD`, with parse failures skipped and validation enforced at the end:

In [ ]:
import tempfile, os
from pathlib import Path
from notebooks.agent.config_loader import load_config, ConfigError

with tempfile.TemporaryDirectory() as d:
    cwd = Path(d)
    # Project-local config overrides the model name and turns.
    (cwd / ".ai-agent").mkdir()
    (cwd / ".ai-agent" / "config.toml").write_text(
        'max_turns = 20\n'
        '[model]\nname = "groq/llama-3.1-70b"\ntemperature = 0.2\n'
    )
    # Developer guide.
    (cwd / "AGENT.MD").write_text("# Guide\nUse snake_case.")

    cfg = load_config(cwd=cwd)

print("loaded model  :", cfg.model.name)
print("loaded temp   :", cfg.model.temperature)
print("loaded turns  :", cfg.max_turns)        # overrode 100 -> 20
print("cwd anchored  :", cfg.cwd == cwd.resolve())
print("dev ins set  :", cfg.developer_instructions is not None)

Note the override flow visible in the output:

- `model.name` became `"groq/llama-3.1-70b"` even though the system *and* the Pydantic default would have left `"anthropic/claude-sonnet-4"`.
- `temperature` was set from the project file (`0.2`), `context_window` (absent from the project file) survives from the system/default (`200_000`).
- `max_turns` shows the project value `20`, not the default `100`.
- `cwd` was anchored at the temp directory even though no TOML file mentioned it — the loader calls `config_dict.setdefault("cwd", cwd)` so the working directory is always meaningful.
- `developer_instructions` came from `AGENT.MD`, not from TOML.

**Validation is the last gate.** `load_config` builds `Config(**config_dict)`, so a typo in the TOML (e.g. an unknown enum value for `approval`) raises `ConfigError`:

In [ ]:
with tempfile.TemporaryDirectory() as d:
    cwd = Path(d)
    (cwd / ".ai-agent").mkdir()
    (cwd / ".ai-agent" / "config.toml").write_text(
        'approval = "always-ask"\n'   # not a valid ApprovalPolicy
    )
    try:
        load_config(cwd=cwd)
    except ConfigError as e:
        print(f"{type(e).__name__}: bad approval enum rejected at validation")

:::{.callout-note}
Corrupt-but-parseable TOML (trailing garbage, missing brackets) and missing files are *skipped with a warning*, not fatal. Only a structurally valid TOML whose values fail Pydantic validation is fatal — because that is a real intent the user expressed, not a degraded environment.

:::

## Wiring the Loader Into the Agent

The `Agent` constructor accepts an optional `Config` and defaults to `Config()` when one is not supplied. To use file-based config, pass the loader's result:

```python
from notebooks.agent import Agent
from notebooks.agent.config_loader import load_config

agent = Agent(config=load_config())
```

That is the entire integration surface — there is no `Agent.from_config_file()` constructor, no hidden kwargs, no global singleton. The loader is just a function that produces a `Config`, and `Agent` already knows what to do with a `Config`.

In [ ]:
from notebooks.agent import Agent
from notebooks.agent.config_loader import load_config

# Empty cwd -> falls back to pure Pydantic defaults, no files on disk.
import tempfile
with tempfile.TemporaryDirectory() as d:
    cfg = load_config(cwd=Path(d))
    agent = Agent(config=cfg)
print(f"agent.config.model_name : {agent.config.model_name}")
print(f"agent.config.max_turns  : {agent.config.max_turns}")
print(f"agent.hooks.hooks       : {agent.hooks.hooks}  (hooks_enabled defaulted to False)")

**Idempotent and side-effect-free.** `load_config` only reads files; it never writes. Calling it twice with the same cwd returns two equivalent `Config` objects. That makes it safe to call from a UI launch path, a test harness, or a CLI entry point without coordination.

## Summary


| Function | Purpose |
|---|---|
| `get_config_dir` / `get_data_dir` | User-scoped paths via `platformdirs` |
| `get_system_config_path` | Full path to `<user_config>/ai-agent/config.toml` |
| `get_project_config_path(cwd)` | Path to `<cwd>/.ai-agent/config.toml` or `None` |
| `get_agent_md_path(cwd)` | Path to `<cwd>/AGENT.MD` or `None` |
| `_parse_toml(path)` | `tomllib.load` wrapped to raise `ConfigError` |
| `_merge_dicts(base, override)` | Deep-merge two dicts (nested dicts recurse) |
| `load_config(cwd=None)` | Assemble system + project + `AGENT.MD`, then validate |

: {tbl-colwidths="[35,65]"}

<br>

← [Hardening the Agent](/notebooks/apps/cda/04-hardening.html) &emsp; → [Hooks System](/notebooks/apps/cda/06-hooks.html)

---

■